# AGENTS026 — GPU-Powered RCA Agent
**AMD Instinct MI300X · Qwen3-30B via vLLM · MiniCluster Banking Stack**

This notebook runs the full autonomous loop:
1. Read live telemetry from `live_metrics.csv`
2. Detect anomalies (threshold + statistical)
3. Call Qwen3-30B on GPU for Root Cause Analysis
4. Decide: auto-remediate or escalate to HITL
5. Execute remediation via fault injection APIs
6. Write results to audit log and HITL queue (visible in Streamlit console)

## Cell 1 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import json, time, uuid, requests
from datetime import datetime, timezone
from pathlib import Path
from openai import OpenAI
from IPython.display import display, HTML, clear_output

# ── paths ──────────────────────────────────────────────────────────────────
BASE          = Path('/workspace/shared')
METRICS_CSV   = BASE / 'minicluster' / 'live_metrics.csv'
HITL_FILE     = BASE / 'hitl_queue.jsonl'
AUDIT_FILE    = BASE / 'audit_log.jsonl'
INCIDENTS_DIR = BASE / 'incidents'
INCIDENTS_DIR.mkdir(parents=True, exist_ok=True)
HITL_FILE.parent.mkdir(parents=True, exist_ok=True)
AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)

# ── vLLM client (GPU) ──────────────────────────────────────────────────────
llm = OpenAI(base_url='http://localhost:8000/v1', api_key='abc-123')
MODEL = 'Qwen3-30B-A3B'

# ── service ports ──────────────────────────────────────────────────────────
SERVICES = {'payments': 7001, 'auth': 7002, 'checkout': 7003, 'fraud': 7004}

# ── thresholds ─────────────────────────────────────────────────────────────
THRESHOLDS = {
    'cpu_utilization': 70.0,
    'latency_p95_ms':  500.0,
    'error_rate':      0.05,
    'mem_mb':          800.0,
}

# ── remediation confidence gate ────────────────────────────────────────────
# confidence >= this → auto-remediate; below → HITL
AUTO_REMEDIATE_THRESHOLD = 0.75

print('✅ Config loaded')
print(f'   Metrics CSV : {METRICS_CSV}')
print(f'   HITL file   : {HITL_FILE}')
print(f'   Audit file  : {AUDIT_FILE}')
print(f'   LLM model   : {MODEL}')

## Cell 2 — Verify GPU / vLLM is alive

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['rocm-smi', '--showuse'], capture_output=True, text=True)
print('=== GPU Status ===')
print(result.stdout or result.stderr)

# Check vLLM
try:
    models = llm.models.list()
    print(f'\n✅ vLLM alive — models: {[m.id for m in models.data]}')
except Exception as e:
    print(f'❌ vLLM not reachable: {e}')
    print('   Make sure Terminal 1 vLLM is running and fully loaded')

## Cell 3 — Helper Functions

In [ ]:
def ts():
    return datetime.now(timezone.utc).isoformat()

def load_metrics(window_mins=10):
    """Load recent metrics from live_metrics.csv"""
    df = pd.read_csv(METRICS_CSV, parse_dates=['timestamp'])
    cutoff = df['timestamp'].max() - pd.Timedelta(minutes=window_mins)
    return df[df['timestamp'] >= cutoff].copy()

def detect_anomalies(df):
    """Threshold + z-score anomaly detection"""
    anomalies = []
    latest = df.sort_values('timestamp').groupby('service').last().reset_index()

    for _, row in latest.iterrows():
        svc = row['service']
        for metric, thresh in THRESHOLDS.items():
            if metric not in row:
                continue
            val = float(row[metric])
            if val > thresh:
                # z-score over recent window for severity
                svc_df = df[df['service'] == svc][metric].dropna()
                z = (val - svc_df.mean()) / (svc_df.std() + 1e-9)
                severity = 'CRITICAL' if z > 3 else ('HIGH' if val > thresh * 1.5 else 'WARN')
                anomalies.append({
                    'service':   svc,
                    'metric':    metric,
                    'value':     round(val, 4),
                    'threshold': thresh,
                    'z_score':   round(float(z), 2),
                    'severity':  severity,
                    'node':      row.get('node', 'unknown'),
                    'timestamp': str(row['timestamp']),
                })
    return anomalies

def fault_post(port, path, payload):
    """Call fault injection API"""
    try:
        r = requests.post(f'http://127.0.0.1:{port}{path}', json=payload, timeout=3)
        return r.json()
    except Exception as e:
        return {'error': str(e)}

def write_hitl(event):
    with open(HITL_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def write_audit(event):
    with open(AUDIT_FILE, 'a') as f:
        f.write(json.dumps(event, default=str) + '\n')

def save_incident(incident):
    path = INCIDENTS_DIR / f"{incident['incident_id']}.json"
    with open(path, 'w') as f:
        json.dump(incident, f, indent=2, default=str)

print('✅ Helper functions loaded')

## Cell 4 — GPU RCA Agent (core LLM call)

In [ ]:
def run_rca_agent(anomalies, metrics_summary):
    """
    Call Qwen3-30B on GPU for root cause analysis.
    Returns structured JSON with: root_cause, confidence, action, auto_remediate
    """
    system_prompt = """You are an autonomous SRE agent for a banking microservices platform.
You diagnose incidents and decide remediation actions.

Services: payments (7001), auth (7002), checkout (7003), fraud (7004)
All run on AMD MI300X GPU cluster.

Fault injection endpoints available:
- POST /fault/latency  {"ms": N}         — adds latency
- POST /fault/errors   {"pct": 0.0-1.0}  — injects error rate  
- POST /fault/cpu_spin {"seconds": N}    — CPU spike
- POST /fault/mem_leak {"mb_per_min": N} — memory leak
- POST /fault/clear    {}                — clears ALL faults on a service

Remediation actions you can take:
- clear_fault: clear all faults on affected service (safe, always auto-approve)
- restart_service: restart the service (medium risk, auto-approve if confidence > 0.8)
- scale_service: scale out (low risk)
- rollback_config: rollback config change (high risk, always send to HITL)
- drain_node: drain the node (high risk, always send to HITL)
- escalate: cannot determine cause, human needed

Respond ONLY with valid JSON, no markdown, no explanation outside JSON:
{
  "root_cause": "one sentence explanation",
  "root_cause_detail": "2-3 sentence technical detail",
  "affected_services": ["service1"],
  "confidence": 0.0-1.0,
  "action": "clear_fault|restart_service|scale_service|rollback_config|drain_node|escalate",
  "action_target": "service_name",
  "action_params": {},
  "auto_remediate": true|false,
  "reasoning": "why this action",
  "risk_level": "LOW|MEDIUM|HIGH"
}"""

    user_msg = f"""ANOMALIES DETECTED:
{json.dumps(anomalies, indent=2)}

RECENT METRICS SUMMARY:
{metrics_summary}

Diagnose the root cause and decide on remediation action."""

    print('🧠 Calling Qwen3-30B on GPU for RCA...')
    t0 = time.time()

    response = llm.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_msg}
        ],
        temperature=0.1,
        max_tokens=800,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}}
    )

    elapsed = time.time() - t0
    raw = response.choices[0].message.content.strip()
    tokens = response.usage.completion_tokens
    print(f'   ✅ LLM responded in {elapsed:.1f}s — {tokens} tokens ({tokens/elapsed:.0f} tok/s)')

    # parse JSON
    try:
        # strip any accidental markdown
        clean = raw.replace('```json','').replace('```','').strip()
        result = json.loads(clean)
    except json.JSONDecodeError:
        print(f'   ⚠️  JSON parse failed, raw: {raw[:200]}')
        result = {
            'root_cause': 'LLM response parse error',
            'confidence': 0.0,
            'action': 'escalate',
            'auto_remediate': False,
            'risk_level': 'HIGH',
            'raw_response': raw
        }

    result['llm_latency_s']  = round(elapsed, 2)
    result['tokens_used']    = tokens
    return result

print('✅ RCA agent function loaded')

## Cell 5 — Remediation Executor

In [ ]:
def execute_remediation(rca, incident_id):
    """
    Execute the RCA agent's recommended action.
    Returns execution result dict.
    """
    action  = rca.get('action', 'escalate')
    target  = rca.get('action_target', '')
    params  = rca.get('action_params', {})
    port    = SERVICES.get(target)

    result = {
        'incident_id': incident_id,
        'action':      action,
        'target':      target,
        'timestamp':   ts(),
        'status':      'UNKNOWN',
        'detail':      ''
    }

    if action == 'clear_fault' and port:
        r = fault_post(port, '/fault/clear', {})
        result['status'] = 'EXECUTED'
        result['detail'] = f'Cleared all faults on {target}: {r}'
        print(f'   🔧 Cleared faults on {target}: {r}')

    elif action == 'restart_service' and port:
        # Simulate restart via clear + brief delay
        r = fault_post(port, '/fault/clear', {})
        result['status'] = 'EXECUTED'
        result['detail'] = f'Simulated restart of {target} (clear + reset): {r}'
        print(f'   🔄 Restarted {target}: {r}')

    elif action == 'scale_service':
        result['status'] = 'SIMULATED'
        result['detail'] = f'Scale-out of {target} simulated (no real infra API)'
        print(f'   📈 Scale-out {target} simulated')

    elif action in ('rollback_config', 'drain_node', 'escalate'):
        result['status'] = 'SKIPPED'
        result['detail'] = f'High-risk action {action} — sent to HITL instead'
        print(f'   ⚠️  {action} is high-risk — routing to HITL')

    else:
        result['status'] = 'UNKNOWN_ACTION'
        result['detail'] = f'No executor for action: {action}'
        print(f'   ❓ Unknown action: {action}')

    return result

print('✅ Remediation executor loaded')

## Cell 6 — Full Autonomous Loop (single run)

In [ ]:
def run_autonomous_loop(verbose=True):
    """
    One full iteration:
    detect → RCA (GPU) → decide → execute or HITL → audit
    """
    incident_id = f'inc-{uuid.uuid4().hex[:8]}'
    loop_start  = time.time()

    print(f'\n{"="*60}')
    print(f'🔁 AUTONOMOUS LOOP — {incident_id}')
    print(f'   {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'{"="*60}')

    # ── Step 1: load metrics ───────────────────────────────────────────────
    df = load_metrics(window_mins=10)
    if df.empty:
        print('⚠️  No metrics data yet — is the collector running?')
        return None

    latest = df.sort_values('timestamp').groupby('service').last()
    metrics_summary = latest[['cpu_utilization','latency_p95_ms',
                               'error_rate','mem_mb']].round(3).to_string()
    print(f'\n📊 Latest metrics:\n{metrics_summary}')

    # ── Step 2: detect anomalies ───────────────────────────────────────────
    anomalies = detect_anomalies(df)
    print(f'\n🔍 Anomalies detected: {len(anomalies)}')

    if not anomalies:
        print('   ✅ All services healthy — no action needed')
        write_audit({
            'event_type':   'LOOP_RUN_HEALTHY',
            'incident_id':  incident_id,
            'timestamp':    ts(),
            'anomaly_count': 0
        })
        return {'status': 'healthy', 'incident_id': incident_id}

    for a in anomalies:
        sev_icon = '🔴' if a['severity'] == 'CRITICAL' else ('🟡' if a['severity'] == 'HIGH' else '⚪')
        print(f'   {sev_icon} {a["service"]}.{a["metric"]} = {a["value"]} '
              f'(thresh={a["threshold"]}, z={a["z_score"]}, {a["severity"]})')

    # ── Step 3: GPU RCA ────────────────────────────────────────────────────
    print(f'\n🧠 Running GPU-based RCA...')
    rca = run_rca_agent(anomalies, metrics_summary)

    print(f'\n📋 RCA Result:')
    print(f'   Root cause  : {rca.get("root_cause")}')
    print(f'   Confidence  : {rca.get("confidence")}')
    print(f'   Action      : {rca.get("action")} → {rca.get("action_target")}')
    print(f'   Risk level  : {rca.get("risk_level")}')
    print(f'   Auto-fix    : {rca.get("auto_remediate")}')
    print(f'   Reasoning   : {rca.get("reasoning")}')

    # ── Step 4: decide — auto or HITL ─────────────────────────────────────
    confidence     = float(rca.get('confidence', 0))
    auto_remediate = rca.get('auto_remediate', False)
    risk_level     = rca.get('risk_level', 'HIGH')

    # Force HITL for high-risk actions regardless of confidence
    if risk_level == 'HIGH' or rca.get('action') in ('rollback_config','drain_node','escalate'):
        auto_remediate = False

    # Force HITL if confidence too low
    if confidence < AUTO_REMEDIATE_THRESHOLD:
        auto_remediate = False

    exec_result = None

    if auto_remediate:
        print(f'\n⚡ AUTO-REMEDIATE (confidence={confidence:.2f} >= {AUTO_REMEDIATE_THRESHOLD})')
        exec_result = execute_remediation(rca, incident_id)
        write_audit({
            'event_type':    'AUTO_REMEDIATION',
            'incident_id':   incident_id,
            'timestamp':     ts(),
            'anomalies':     anomalies,
            'rca':           rca,
            'exec_result':   exec_result,
        })
        print(f'   Status: {exec_result["status"]} — {exec_result["detail"]}')
    else:
        print(f'\n🛑 ESCALATING TO HITL '
              f'(confidence={confidence:.2f}, risk={risk_level}, auto={auto_remediate})')
        hitl_event = {
            'hitl_id':     f'hitl-{uuid.uuid4().hex[:8]}',
            'incident_id': incident_id,
            'timestamp':   ts(),
            'source':      'rca_agent',
            'anomalies':   anomalies,
            'rca':         rca,
            'status':      'PENDING',
            'operator':    None,
        }
        write_hitl(hitl_event)
        write_audit({**hitl_event, 'event_type': 'HITL_CREATED'})
        print(f'   ✅ Written to HITL queue — check Streamlit console HITL tab')

    # ── Step 5: save incident ──────────────────────────────────────────────
    incident = {
        'incident_id':   incident_id,
        'timestamp':     ts(),
        'anomalies':     anomalies,
        'rca':           rca,
        'exec_result':   exec_result,
        'auto_remediated': auto_remediate,
        'loop_duration_s': round(time.time() - loop_start, 2),
    }
    save_incident(incident)

    print(f'\n✅ Loop complete in {incident["loop_duration_s"]}s')
    print(f'   Incident saved: {INCIDENTS_DIR}/{incident_id}.json')
    return incident

print('✅ Autonomous loop function loaded')

## Cell 7 — Inject a Fault + Run Loop (Demo)

In [ ]:
# Step 1: Inject a real fault into payments service
print('💥 Injecting fault: 800ms latency + 20% errors on payments...')
r1 = requests.post('http://127.0.0.1:7001/fault/latency', json={'ms': 800}, timeout=3)
r2 = requests.post('http://127.0.0.1:7001/fault/errors',  json={'pct': 0.2}, timeout=3)
print(f'   Latency: {r1.json()}')
print(f'   Errors : {r2.json()}')

print('\n⏳ Waiting 70s for metrics to capture the fault...')
for i in range(70, 0, -10):
    print(f'   {i}s remaining...')
    time.sleep(10)

print('\n🚀 Running autonomous loop now...')
result = run_autonomous_loop()
print('\n📊 Final result:')
print(json.dumps(result, indent=2, default=str))

## Cell 8 — Continuous Watch Loop (runs until stopped)

In [ ]:
# ⚠️  This cell runs continuously — Kernel > Interrupt to stop
POLL_INTERVAL = 60  # seconds between checks
loop_count = 0

print(f'🔄 Starting continuous watch loop (every {POLL_INTERVAL}s)')
print(f'   Kernel > Interrupt Kernel to stop')
print(f'   Results stream to Streamlit console in real time')
print(f'   HITL items appear immediately in the HITL Queue tab')

try:
    while True:
        loop_count += 1
        clear_output(wait=True)
        print(f'🔄 Continuous watch loop — iteration {loop_count}')
        print(f'   Poll interval: {POLL_INTERVAL}s | Ctrl+C or Kernel Interrupt to stop')
        result = run_autonomous_loop()
        print(f'\n💤 Next check in {POLL_INTERVAL}s...')
        time.sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    print(f'\n⛔ Watch loop stopped after {loop_count} iterations')

## Cell 9 — Manual: Inject Specific Faults for Testing

In [ ]:
# Run any of these individually to test different scenarios

# Scenario A — High latency on auth (should auto-remediate)
# requests.post('http://127.0.0.1:7002/fault/latency', json={'ms': 1200}, timeout=3)

# Scenario B — High error rate on checkout (HITL escalation)
# requests.post('http://127.0.0.1:7003/fault/errors', json={'pct': 0.5}, timeout=3)

# Scenario C — CPU spike on fraud (HITL — high risk)
# requests.post('http://127.0.0.1:7004/fault/cpu_spin', json={'seconds': 90}, timeout=3)

# Scenario D — Memory leak on payments
# requests.post('http://127.0.0.1:7001/fault/mem_leak', json={'mb_per_min': 100}, timeout=3)

# Clear all faults
# for port in [7001,7002,7003,7004]:
#     requests.post(f'http://127.0.0.1:{port}/fault/clear', json={}, timeout=3)

# Run one manual loop without waiting
result = run_autonomous_loop()
print(json.dumps({
    'incident_id':     result.get('incident_id'),
    'anomaly_count':   len(result.get('anomalies',[])),
    'root_cause':      result.get('rca',{}).get('root_cause'),
    'action':          result.get('rca',{}).get('action'),
    'auto_remediated': result.get('auto_remediated'),
    'duration_s':      result.get('loop_duration_s'),
}, indent=2))

## Cell 10 — View Audit Log & HITL Queue

In [ ]:
print('=== AUDIT LOG (last 5 events) ===')
if AUDIT_FILE.exists():
    lines = AUDIT_FILE.read_text().strip().split('\n')
    for line in lines[-5:]:
        evt = json.loads(line)
        print(f"  [{evt.get('timestamp','')}] {evt.get('event_type')} — "
              f"{evt.get('incident_id',evt.get('hitl_id',''))}")
else:
    print('  No audit log yet')

print('\n=== HITL QUEUE ===')
if HITL_FILE.exists():
    lines = HITL_FILE.read_text().strip().split('\n')
    for line in lines:
        if not line.strip():
            continue
        h = json.loads(line)
        status_icon = '⏳' if h['status']=='PENDING' else ('✅' if h['status']=='APPROVED' else '❌')
        print(f"  {status_icon} {h.get('hitl_id')} | {h.get('status')} | "
              f"{h.get('rca',{}).get('action','?')} → {h.get('rca',{}).get('action_target','?')}")
else:
    print('  No HITL items yet')